# Learner Persona Detection — Interactive Exploration

This notebook walks through the full pipeline for identifying **behavioral learner personas** from online learning activity data.

The goal is to cluster students into distinct profiles that can guide **adaptive content design** — helping instructional designers create the right type of content for each type of learner.

---
**Dataset:** Synthetic data modeled on [OULAD](https://analyse.kmi.open.ac.uk/open_dataset) (Open University Learning Analytics Dataset)  
**Method:** K-Means Clustering + PCA Visualization  
**Output:** 4 learner personas with content recommendations

## 1. Setup & Data Generation

In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'data'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

from simulate_oulad import simulate_oulad

PALETTE = ['#4C72B0', '#55A868', '#C44E52', '#CCB974']
PERSONA_NAMES = {
    0: 'Independent Deep Learner',
    1: 'Guided Learner',
    2: 'Exam-Driven Learner',
    3: 'At-Risk / Disengaged'
}

df = simulate_oulad(n=400)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

Before clustering, we look at the distribution of key behavioral features.

In [ ]:
df.describe().round(2)

In [ ]:
FEATURES = [
    'total_clicks', 'active_days', 'avg_session_min',
    'forum_posts', 'resource_clicks', 'video_clicks',
    'quiz_attempts', 'avg_quiz_score',
    'assignment_submit_lag', 'help_requests', 'login_freq_per_week'
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

for i, (ax, feat) in enumerate(zip(axes.flat, FEATURES)):
    ax.hist(df[feat], bins=25, color=PALETTE[i % 4], alpha=0.8, edgecolor='white')
    ax.set_title(feat, fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
corr = df[FEATURES].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Pearson r'})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Preprocessing

K-Means is distance-based, so all features must be on the same scale.  
We apply **StandardScaler** (zero mean, unit variance).

In [ ]:
X = df[FEATURES].copy()
y_true = df['latent_type'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Mean after scaling (should be ~0):', X_scaled.mean(axis=0).round(2))
print('Std  after scaling (should be ~1):', X_scaled.std(axis=0).round(2))

## 4. Choosing K — Elbow Method & Silhouette Score

We test K from 2 to 9 and look for:
- **Elbow point** in inertia (where adding more clusters gives diminishing returns)
- **Peak** in silhouette score (how well-separated the clusters are)

In [ ]:
K_range = range(2, 10)
inertias, sil_scores = [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Selecting Optimal Number of Clusters', fontsize=13, fontweight='bold')

axes[0].plot(K_range, inertias, 'o-', color=PALETTE[0], linewidth=2)
axes[0].axvline(4, color=PALETTE[2], linestyle='--', alpha=0.7, label='K = 4')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method'); axes[0].legend()

axes[1].plot(K_range, sil_scores, 's-', color=PALETTE[1], linewidth=2)
axes[1].axvline(4, color=PALETTE[2], linestyle='--', alpha=0.7, label='K = 4')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis'); axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Best silhouette score: {max(sil_scores):.3f} at K={list(K_range)[sil_scores.index(max(sil_scores))]}')

Both methods agree on **K = 4**, which also aligns with our prior knowledge about learner diversity in online education.

## 5. Fitting Final Model (K = 4)

In [ ]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=20)
cluster_labels = km_final.fit_predict(X_scaled)
df['cluster'] = cluster_labels

sil  = silhouette_score(X_scaled, cluster_labels)
ari  = adjusted_rand_score(y_true, cluster_labels)

print(f'Silhouette Score            : {sil:.3f}')
print(f'Adjusted Rand Index         : {ari:.3f}')
print(f'\nCluster sizes:')
print(df['cluster'].value_counts().sort_index())

## 6. Visualizations

In [ ]:
# PCA projection
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(9, 6))
for c in range(4):
    mask = cluster_labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=PALETTE[c], label=PERSONA_NAMES[c],
               alpha=0.75, edgecolors='white', s=60)

ax.set_xlabel(f'PC 1 ({var[0]:.1f}% variance)')
ax.set_ylabel(f'PC 2 ({var[1]:.1f}% variance)')
ax.set_title('Learner Personas — PCA Projection', fontsize=13, fontweight='bold')
ax.legend(title='Persona', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Feature heatmap
heatmap_data = df.groupby('cluster')[FEATURES].mean()
heatmap_norm = (heatmap_data - heatmap_data.mean()) / heatmap_data.std()
heatmap_norm.index = [PERSONA_NAMES[i] for i in range(4)]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap_norm, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Z-score'})
ax.set_title('Feature Z-scores per Learner Persona', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Persona Interpretation & Content Recommendations

Based on the cluster centroids, each group shows a distinct behavioral signature.

In [ ]:
CONTENT_REC = {
    0: 'Research papers, open-ended projects, peer discussion forums.',
    1: 'Step-by-step video tutorials, worked examples, immediate-feedback quizzes.',
    2: 'Concise summaries, flashcards, practice exams, spaced-repetition tools.',
    3: 'Short micro-content, gamification, push notifications, mentor check-ins.'
}

summary = df.groupby('cluster')[FEATURES].mean().round(1)
summary['n_students'] = df['cluster'].value_counts().sort_index()
summary['persona'] = [PERSONA_NAMES[i] for i in range(4)]
summary['content_recommendation'] = [CONTENT_REC[i] for i in range(4)]

for c in range(4):
    n = (cluster_labels == c).sum()
    print(f'Cluster {c} — {PERSONA_NAMES[c]} (n={n}, {100*n/len(cluster_labels):.0f}%)')
    print(f'  → {CONTENT_REC[c]}')
    print()

## 8. Export Results

In [ ]:
df['persona'] = df['cluster'].map(PERSONA_NAMES)
df.to_csv('../learner_profiles.csv')
print('Saved → learner_profiles.csv')
df[['avg_quiz_score', 'total_clicks', 'active_days', 'cluster', 'persona']].head(10)

---
## Next Steps

- Validate on real OULAD data (download from [here](https://analyse.kmi.open.ac.uk/open_dataset))
- Try **Gaussian Mixture Models** for soft cluster membership
- Model persona transitions over time (longitudinal analysis)
- Frame as a **reinforcement learning** problem: an agent that selects content type based on evolving student state